# Station PM2.5 sub-index forecaster (CPCB stations, h = 1-3 days)

Trains `aqi_station_xgb_v1` on the private Kaggle dataset `shaikhfardin05/india-aqi-station-pm25`, produced by
`ml/pipeline/build_aqi_station_dataset.py`. Target: CPCB PM2.5 sub-index of cleaned CPCB station daily PM2.5 (OpenAQ v3).
Two variants: `with_station` (station lags + CAMS + weather) and `no_station` (CAMS + weather + calendar + location).

The notebook only calls `train_aqi_station.py` (shipped inside the dataset, a copy of `ml/train/train_aqi_station.py`), so
Kaggle and local runs execute identical code. Tuning: train t+h <= 2025-06-30, val 2025-07-01..2025-10-31. Test
2025-11-01..2026-10-28 scored once with a monthly expanding window (refit only on rows with t+h before each month).
Runs on GPU T4 x2: every XGBoost fit runs in two worker processes pinned to `cuda:0` and `cuda:1`;
the CPU only loads data and computes baselines and metrics. Outputs land in `/kaggle/working`.

In [ ]:
import glob
import subprocess
import sys

data = glob.glob("/kaggle/input/**/aqi_station_dataset.parquet", recursive=True)[0]
script = glob.glob("/kaggle/input/**/train_aqi_station.py", recursive=True)[0]
v1 = glob.glob("/kaggle/input/**/aqi_pm25_v1_test_predictions.parquet", recursive=True)[0]
print("dataset:", data, "script:", script, "v1:", v1, flush=True)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout, flush=True)
# The script's output is piped and re-printed here, so each line reaches the log once.
cmd = [
    sys.executable,
    "-u",
    script,
    "--dataset",
    data,
    "--v1-predictions",
    v1,
    "--out-dir",
    "/kaggle/working",
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError(f"train_aqi_station.py exited with {proc.returncode}")

In [ ]:
import json

ev = json.load(open("/kaggle/working/aqi_station_evaluation.json"))
print(json.dumps({k: ev[k] for k in ("protocol", "leakage_checks")}, indent=2))
for h, t in ev["test"].items():
    print(
        h,
        json.dumps(
            {k: t[k].get("overall") for k in ("table_with_station", "table_no_station")}, indent=1
        ),
    )